# Multi-Agent Manipulation Detection System

Notebook này sử dụng hệ thống multi-agent với LangGraph để phát hiện manipulation trong các cuộc đối thoại pháp lý.

## Kiến trúc hệ thống:
1. **Detector Agent**: Phát hiện có manipulation hay không
2. **Analyzer Agent**: Xác định manipulator và kỹ thuật
3. **BERT Technique Agent**: Dự đoán kỹ thuật thao túng bằng mô hình Legal-BERT
5. **Meta Agent**: Tổng hợp kết quả cuối cùng

## Model:
### LLM
- Base: Mistral-7B-Instruct-v0.3 (4-bit quantized)
- Fine-tuned: Custom adapter từ stage 1

### Technique Classification Model:
-  Encoder: nlpaueb/legal-bert-base-uncased
-  Task: Multi-label classification cho 11 kỹ thuật thao túng

## 1. Cài đặt thư viện

In [1]:
!pip install -U transformers peft accelerate langgraph bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 90.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## 2. Import thư viện

In [2]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import json 
import os
import pandas as pd 
import torch 
import torch.nn as nn
import numpy as np
from transformers import AutoModel, AutoTokenizer
from typing import Dict, Any, Optional, List, Union, TypedDict 
import re
from langchain_core.messages import HumanMessage, AIMessage 
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import MemorySaver 
from langgraph.graph import END, START, StateGraph 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer


## 3. Cấu hình HuggingFace Token

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

## 4. Kiểm tra môi trường

In [4]:
print("=== Environment Check ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")
print("=" * 50)

=== Environment Check ===
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 15.64 GB
PyTorch Version: 2.6.0+cu124


## 5. Load Base Model

Tải Mistral-7B-Instruct với cấu hình với cấu hình quantization 4-bit

In [5]:
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer from {base_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=HF_TOKEN
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")

print(f"Loading base model from {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

base_model.config.pad_token_id = tokenizer.pad_token_id

print("Base model loaded in 4-bit")

Loading tokenizer from mistralai/Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded in 4-bit


## 6. Load Fine-tuned Adapter & Merge

Load PEFT adapter đã được fine-tune từ stage 1

In [6]:
adapter_model_id = "/kaggle/input/datasets/banabag/raw-finetune-mistral/final_model_best"  
if not os.path.exists(adapter_model_id):
    raise FileNotFoundError(f"Adapter path not found: {adapter_model_id}")

print(f"Found adapter at: {adapter_model_id}")

try:
    model = PeftModel.from_pretrained(base_model, adapter_model_id)
    print("PEFT adapter loaded successfully")
    
except Exception as e:
    print(f"Failed to load/merge PEFT adapter: {e}")
    raise

Found adapter at: /kaggle/input/datasets/banabag/raw-finetune-mistral/final_model_best
PEFT adapter loaded successfully


## 7. Load Test Data

Load dataset chứa:
- **Dialogue**: Cuộc hội thoại giữa plaintiff và defendant
- **PLAINTIFF intent**: Ý định của nguyên đơn
- **DEFENDANT intent**: Ý định của bị đơn

In [7]:
def load_csv(file_path):
    return pd.read_csv(file_path)

csv_df = load_csv("/kaggle/input/datasets/linhtron123/test-intent-raw/test_intent_raw.csv")
dialogues = csv_df["Dialogue"].tolist()
plaintiff_intents = csv_df["PLAINTIFF"].tolist()
defendant_intents = csv_df["DEFENDANT"].tolist()

## 8. Định nghĩa Manipulation Tactics

Danh sách 11 kỹ thuật manipulation được model nhận diện

In [8]:
allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion", 
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

## 9. Hàm Generation

Hàm sinh text từ model

In [9]:
def generate_response(prompt, max_new_tokens=1000):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 10. Legal-Bert Technique Classifer

Mô hình Legal-BERT được sử dụng cho bài toán multi-label classification nhằm nhận diện các kỹ thuật thao túng


In [10]:
class ManipulationClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.35, k_layers=3):
        super().__init__()
        self.k_layers = k_layers
        self.encoder = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True  
        )
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.LayerNorm(hidden_size // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )

    def max_cls_pool(self, hidden_states):
        cls_per_layer = torch.stack(
            [hidden_states[-(i + 1)][:, 0, :] for i in range(self.k_layers)],
            dim=1
        )  # (B, k, H)
        return cls_per_layer.max(dim=1).values  # (B, H)

    def forward(self, input_ids, attention_mask, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.max_cls_pool(out.hidden_states)
        logits = self.classifier(pooled)
        return None, logits

In [11]:
class TechniqueClassifier:
    def __init__(self, model_path, optuna_path, device="cuda"):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        
        with open(optuna_path, "r") as f:
            best_params = json.load(f)["best_params"]

        self.model = ManipulationClassifier(
            "nlpaueb/legal-bert-base-uncased",
            num_labels=11,
            dropout=best_params["dropout"],
            k_layers=3 
        )
        self.model.load_state_dict(torch.load(f"{model_path}/best_model.pt", map_location=device))
        self.model.to(device)
        self.model.eval()
        
        self.thresholds = np.load(f"{model_path}/best_thresholds.npy")
        self.techniques_list = [
            "persuasion", "playing the victim", "gaslighting", "evasion", 
            "deflection", "minimization", "dismissal", "guilt tripping",
            "emotional appeal", "framing the narrative", "character attack"
        ]
        print(f"  [BERT] Model loaded | thresholds: {[round(t,2) for t in self.thresholds]}")

    def predict(self, dialogue, plaintiff_intent="", defendant_intent=""):
        if plaintiff_intent and defendant_intent:
            text = (f"[PLAINTIFF INTENT] {plaintiff_intent} "
                    f"[DEFENDANT INTENT] {defendant_intent} "
                    f"[DIALOGUE] {dialogue}")
        else:
            text = dialogue

        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        chunks_input_ids      = []
        chunks_attention_mask = []

        for start in range(0, max(len(tokens), 1), 256):
            end   = min(start + 512 - 2, len(tokens))
            chunk = [self.tokenizer.cls_token_id] + tokens[start:end] + [self.tokenizer.sep_token_id]

            pad_len        = 512 - len(chunk)
            attention_mask = [1] * len(chunk) + [0] * pad_len
            chunk          = chunk + [self.tokenizer.pad_token_id] * pad_len

            chunks_input_ids.append(chunk)
            chunks_attention_mask.append(attention_mask)

            if end == len(tokens):
                break

        input_ids      = torch.tensor(chunks_input_ids).to(self.device)
        attention_mask = torch.tensor(chunks_attention_mask).to(self.device)

        with torch.no_grad():
            _, logits = self.model(input_ids, attention_mask)
            probs     = torch.sigmoid(logits).cpu().numpy()

        final_probs = probs.max(axis=0)
        print("\n[DEBUG BERT RAW CONFIDENCE]")
        for i, p in enumerate(final_probs):
            tech = self.techniques_list[i]
            tau  = self.thresholds[i]
            print(f"{tech:25s} | prob={p:.3f} | threshold={tau:.3f}")

        
        pred_techniques = [
            self.techniques_list[i] for i, p in enumerate(final_probs)
            if p >= self.thresholds[i]
        ]
        confidences = {self.techniques_list[i]: float(final_probs[i]) for i in range(len(final_probs))}
        return pred_techniques, confidences


BERT_CLASSIFIER = TechniqueClassifier(
    model_path="/kaggle/input/datasets/anhtuan23004/no-intent-legalbert/bert_final_no_intent",
    optuna_path="/kaggle/input/datasets/anhtuan23004/no-intent-legalbert/bert_final_no_intent/optuna_results.json"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

  [BERT] Model loaded | thresholds: [0.53, 0.58, 0.58, 0.45, 0.5, 0.6, 0.5, 0.5, 0.48, 0.5, 0.4]


## 11. Agent State Definition
AgentState định nghĩa trạng thái được chia sẻ giữa các agent trong workflow.

In [12]:
class AgentState(TypedDict):
    dialogue:              str
    dialogue_index:        int
    plaintiff_intent:      str
    defendant_intent:      str
    manipulation_detected: Optional[str]
    primary_manipulator:   Optional[str]
    evidence:              Optional[str]
    final_summary:         Optional[Dict[str, Any]]
    messages:              List[Union[HumanMessage, AIMessage]]
    techniques_llm:        Optional[List[str]]
    techniques_bert:       Optional[List[str]]
    bert_confidences:      Optional[Dict[str, float]]
    bert_failed:           Optional[bool]
    techniques_used:       Optional[List[str]]

## 12. Detector Agent

**Agent đầu tiên** trong pipeline:
- **Nhiệm vụ**: Phát hiện có manipulation hay không
- **Output**: Yes/No
- **Routing**: Yes → Analyzer Agent, No → Meta Agent (skip analysis)

In [13]:
def detector_agent(state):
    dialogue         = state["dialogue"]
    plaintiff_intent = state["plaintiff_intent"]
    defendant_intent = state["defendant_intent"]

    system_prompt = f"""<s>[INST] You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.
### Instructions:
1. Read the dialogue carefully.
2. Analyze it with the context of the intents provided
3. Determine if any manipulation is present or not.
Defendant's Intent: {defendant_intent}
Plaintiff's Intent: {plaintiff_intent}
Dialogue: {dialogue}
Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: Yes or No
[/INST]"""

    response = generate_response(system_prompt, max_new_tokens=5)

    model_response = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    extracted_response = model_response.group(1).strip() if model_response else response.strip()

    lines = [line.strip() for line in extracted_response.split('\n') if line.strip()]
    if lines:
        last_line = lines[-1]
        if last_line.lower() == "yes":
            result = "Yes"
            status = "Yes (exact match)"
        elif last_line.lower() == "no":
            result = "No"
            status = "No (exact match)"
        else:
            result = "No"
            status = f"No (default, got: '{last_line}')"
    else:
        result = "No"
        status = "No (default, empty response)"

    print(f"  [Detector] raw='{extracted_response.strip()}' → {status}")

    return {
        **state,
        "manipulation_detected": result,
        "messages": state["messages"] + [AIMessage(content=f"Manipulation Present: {result}")]
    }

## 13. Analyzer Agent

**Agent thứ hai** (chỉ chạy nếu có manipulation):
- **Nhiệm vụ**: 
  - Xác định primary manipulator (Plaintiff/Defendant)
  - Nhận diện manipulation techniques được sử dụng
- **Input**: Dialogue + intents của cả hai bên
- **Output**: Primary manipulator + list of techniques

In [14]:
def analyzer_agent(state):
    if state["manipulation_detected"] == "No":
        return {
            **state,
            "primary_manipulator": "None",
            "techniques_llm":      [],
            "messages": state["messages"] + [AIMessage(content="No manipulation detected, skipping analysis.")]
        }

    print("Manipulation detected, analyzing manipulator and techniques...")

    combined_prompt = f"""<s>[INST] Compare both dialogue and the given intents of both defendant and plaintiff, and analyze the manipulation:
        Defendant's Intent: {state['defendant_intent']}
        Plaintiff's Intent: {state['plaintiff_intent']}
        Dialogue: {state['dialogue']}
        First, determine who is the primary manipulator (answer with ONLY Defendant or Plaintiff).
        Second, identify which manipulation techniques they used from this list:
        {', '.join(allowed_tactics)}
        Format your response exactly like this:
        Primary Manipulator: [ ]
        Techniques: [list ONLY the techniques that were actually used, separated by commas]
   [/INST]"""

    combined_response = generate_response(combined_prompt, max_new_tokens=100)

    primary_manipulator = "Unknown"
    if "primary manipulator: plaintiff" in combined_response.lower():
        primary_manipulator = "Plaintiff"
    elif "primary manipulator: defendant" in combined_response.lower():
        primary_manipulator = "Defendant"

    techniques_llm = []
    for line in combined_response.split('\n'):
        if line.lower().startswith("techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            for technique in techniques_part.split(','):
                technique = technique.strip()
                for allowed_tactic in allowed_tactics:
                    if technique.lower() == allowed_tactic.lower():
                        techniques_llm.append(allowed_tactic)
                        break

    print(f"  [Analyzer] Manipulator: {primary_manipulator} | Techniques(LLM): {techniques_llm or 'None'}")

    return {
        **state,
        "primary_manipulator": primary_manipulator,
        "techniques_llm":      techniques_llm,
        "messages": state["messages"] + [AIMessage(content=f"Primary Manipulator: {primary_manipulator}\nTechniques: {techniques_llm}")]
    }

## 14. BERT Technique Agent

**Agent thứ ba** - Agent này sử dụng Legal-BERT để dự đoán manipulation techniques độc lập với LLM.


In [15]:
def bert_technique_agent(state):
    if state["manipulation_detected"] == "No":
        return {
            **state,
            "techniques_bert":  [],
            "bert_confidences": {},
            "bert_failed":      False,
            "evidence": "No manipulation detected."
        }

    techniques, confidence = BERT_CLASSIFIER.predict(
        dialogue         = state["dialogue"],
        plaintiff_intent = state.get("plaintiff_intent"),
        defendant_intent = state.get("defendant_intent")
    )

    max_conf    = max(confidence.values()) if confidence else 0
    bert_failed = max_conf < 0.05

    if bert_failed:
        print(f"  [BERT] FAILED (max_conf={max_conf:.4f}) → crosscheck agent sẽ xử lý")
    else:
        print(f"  [BERT] {techniques if techniques else 'None'} (max_conf={max_conf:.3f})")

    evidence  = f"BERT predictions: {techniques}\n"
    evidence += f"BERT failed: {bert_failed}\n"
    evidence += "Confidence: " + str({k: round(v, 3) for k, v in confidence.items() if v > 0.2})

    return {
        **state,
        "techniques_bert":  techniques,   
        "bert_confidences": confidence,
        "bert_failed":      bert_failed,
        "evidence":         evidence,
        "messages": state["messages"] + [AIMessage(content=evidence)]
    }

## 15. Meta Agent

In [16]:
INVALID_MANIPULATORS = {"None", "Unknown", "nan", "none", "unknown"}

In [17]:
def meta_agent(state):
    det_present     = state.get("manipulation_detected", "No")
    pri_manipulator = state.get("primary_manipulator", "None") or "None"
    bert_techs      = state.get("techniques_bert", []) or []
    llm_techs       = state.get("techniques_llm", []) or []
    bert_failed     = state.get("bert_failed", False)
 
    has_manipulation = (det_present == "Yes")
    has_manipulator  = pri_manipulator not in INVALID_MANIPULATORS
 
    if has_manipulation and has_manipulator:
        if not bert_failed and len(bert_techs) > 0:
            # BERT
            final_verdict     = "Yes"
            final_manipulator = pri_manipulator
            final_tech_list   = bert_techs
            case_note         = "BERT Direct Output"
        elif len(llm_techs) > 0:
            # Fallback LLM (BERT failed)
            final_verdict     = "Yes"
            final_manipulator = pri_manipulator
            final_tech_list   = llm_techs
            case_note         = "LLM Fallback (BERT Failed)"
        else:
            final_verdict     = "No"
            final_manipulator = "None"
            final_tech_list   = []
            case_note         = "No techniques found by both models"
    else:
        final_verdict     = "No"
        final_manipulator = "None"
        final_tech_list   = []
        case_note         = "No manipulation detected"
 
    print(f"  [Meta] {case_note} | Final: {final_verdict} | {final_manipulator} | {final_tech_list}")
 
    return {
        **state,
        "techniques_used": final_tech_list,
        "final_summary": {
            "manipulation_present": final_verdict,
            "primary_manipulator":  final_manipulator,
            "techniques":           final_tech_list,
        }
    }

## 17. Workflow Graph & Processing Functions

### LangGraph Workflow:
- **StateGraph**: Định nghĩa flow giữa các agents
- **Conditional edges**: Routing logic (Yes/No branching)
- **Entry point**: Detector Agent
- **End point**: Meta Agent

In [18]:
workflow_graph = StateGraph(AgentState)
workflow_graph.add_node("detector_agent",       detector_agent)
workflow_graph.add_node("analyzer_agent",       analyzer_agent)
workflow_graph.add_node("bert_technique_agent", bert_technique_agent)
workflow_graph.add_node("meta_agent",           meta_agent)
 
workflow_graph.set_entry_point("detector_agent")
 
workflow_graph.add_conditional_edges(
    "detector_agent",
    lambda state: "analyzer_agent" if state["manipulation_detected"] == "Yes" else "meta_agent"
)
workflow_graph.add_edge("analyzer_agent",       "bert_technique_agent")
workflow_graph.add_edge("bert_technique_agent", "meta_agent")          
workflow_graph.add_edge("meta_agent",           END)
 
workflow = workflow_graph.compile()

In [19]:
def process_dialogue(dialogue_idx):
    initial_state = AgentState({
        "dialogue":              dialogues[dialogue_idx],
        "dialogue_index":        dialogue_idx,
        "plaintiff_intent":      plaintiff_intents[dialogue_idx],
        "defendant_intent":      defendant_intents[dialogue_idx],
        "manipulation_detected": None,
        "primary_manipulator":   "None",
        "evidence":              None,
        "final_summary":         None,
        "bert_failed":           False,
        "messages": [HumanMessage(content=f"Analyze dialogue {dialogue_idx}")],
        "techniques_llm":        [],
        "techniques_bert":       [],
        "bert_confidences":      {},
        "techniques_used":       [],
    })
    return workflow.invoke(initial_state)

## 18. Run Full Pipeline

In [20]:
def save_results_to_csv():
    manipulation_results    = []
    manipulator_results     = []
    techniques_results      = []
    techniques_llm_results  = []
    techniques_bert_results = []
 
    for idx in range(len(dialogues)):
        print(f"\n=== Dialogue {idx}/{len(dialogues)-1} ===")
        try:
            result              = process_dialogue(idx)
            summary             = result.get('final_summary', {})
            manip_present       = summary.get('manipulation_present', 'No')
            manip_person        = summary.get('primary_manipulator',  'None')
            techs               = summary.get('techniques', [])
            techniques_llm_val  = ", ".join(result.get("techniques_llm",  []) or []) or "None"
            techniques_bert_val = ", ".join(result.get("techniques_bert", []) or []) or "None"
 
        except Exception as e:
            import traceback
            print(f"  [ERROR] Dialogue {idx}:")
            traceback.print_exc()
            manip_present       = 'Error'
            manip_person        = 'Error'
            techs               = []
            techniques_llm_val  = 'Error'
            techniques_bert_val = 'Error'
 
        manipulation_results.append(manip_present)
        manipulator_results.append(manip_person)
        techniques_results.append(', '.join(techs) if techs else 'None')
        techniques_llm_results.append(techniques_llm_val)
        techniques_bert_results.append(techniques_bert_val)
 
        print(f"  → {manip_present} | {manip_person} | {', '.join(techs) if techs else 'None'}")
 
        if idx % 20 == 0 and idx > 0:
            pd.DataFrame({
                'Dialogue_Index':       list(range(idx + 1)),
                'Dialogue':             dialogues[:idx + 1],
                'Manipulation_Present': manipulation_results,
                'Primary_Manipulator':  manipulator_results,
                'Techniques_Used':      techniques_results,
                'Techniques_LLM':       techniques_llm_results,
                'Techniques_BERT':      techniques_bert_results,
            }).to_csv("/kaggle/working/checkpoint.csv", index=False)
            print(f"  [Checkpoint saved at idx={idx}]")
 
    output_df = csv_df[['Dialogue', 'PLAINTIFF', 'DEFENDANT']].copy()
    output_df.insert(0, 'Dialogue_Index', range(len(output_df)))
    output_df['Manipulation_Present'] = manipulation_results
    output_df['Primary_Manipulator']  = manipulator_results
    output_df['Techniques_Used']      = techniques_results
    output_df['Techniques_LLM']       = techniques_llm_results
    output_df['Techniques_BERT']      = techniques_bert_results
 
    output_path = "/kaggle/working/manipulation_results_raw.csv"
    output_df.to_csv(output_path, index=False)
    print(f"\n[DONE] Results saved to: {output_path}")
    return output_path
 
 
if __name__ == "__main__":
    save_results_to_csv()


=== Dialogue 0/154 ===
  [Detector] raw='No' → No (exact match)
  [Meta] No manipulation detected | Final: No | None | []
  → No | None | None

=== Dialogue 1/154 ===
  [Detector] raw='Yes' → Yes (exact match)
Manipulation detected, analyzing manipulator and techniques...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1445 > 512). Running this sequence through the model will result in indexing errors


  [Analyzer] Manipulator: Defendant | Techniques(LLM): None

[DEBUG BERT RAW CONFIDENCE]
persuasion                | prob=0.188 | threshold=0.525
playing the victim        | prob=0.186 | threshold=0.575
gaslighting               | prob=0.144 | threshold=0.575
evasion                   | prob=0.130 | threshold=0.450
deflection                | prob=0.101 | threshold=0.500
minimization              | prob=0.184 | threshold=0.600
dismissal                 | prob=0.209 | threshold=0.500
guilt tripping            | prob=0.186 | threshold=0.500
emotional appeal          | prob=0.158 | threshold=0.475
framing the narrative     | prob=0.199 | threshold=0.500
character attack          | prob=0.183 | threshold=0.400
  [BERT] None (max_conf=0.209)
  [Meta] No techniques found by both models | Final: No | None | []
  → No | None | None

=== Dialogue 2/154 ===
  [Detector] raw='Yes' → Yes (exact match)
Manipulation detected, analyzing manipulator and techniques...
  [Analyzer] Manipulator: Plaintif

## 19. Evaluation

In [21]:
df_results = pd.read_csv("/kaggle/working/manipulation_results_raw.csv")
df_test = pd.read_csv('/kaggle/input/datasets/linhtron123/raw-data-final/test_split_raw.csv')

test = pd.merge(df_results, df_test, on='Dialogue', how='left')
print("Dataset Size:", len(test))
print(test.head())

Dataset Size: 155
   Dialogue_Index                                           Dialogue  \
0               0  Judge\r\nI understand, but my hypothetical ass...   
1               1  Judge:\nWell, but if -- why -- your voluntary ...   
2               2  Judge: Why did you go?\nPlaintiff: I had nothi...   
3               3  \nJudge:\nTell me what happened.\n\nPlaintiff ...   
4               4  Judge: Is your position— is there any daylight...   

                                           PLAINTIFF  \
0  The Plaintiff's statements suggest that they a...   
1  The Plaintiff's goal or motive behind their wo...   
2  The Plaintiff's statements suggest that their ...   
3  The Plaintiff, Nicole Yarboro, is seeking comp...   
4  The Plaintiff's goal or motive in this courtro...   

                                           DEFENDANT Manipulation_Present  \
0  The Defendant's lawyer is arguing that their c...                   No   
1  The Defendant's goal in reopening the voluntar...      

### Task 1: Manipulation Detection (Binary Classification)

Đánh giá khả năng phát hiện có/không có manipulation.

In [22]:
print("True label distribution:\n", test["Manipulative"].value_counts(dropna=False))
print("\nPredicted label distribution:\n", test["Manipulation_Present"].value_counts(dropna=False))

test["Manipulation_Present"] = test["Manipulation_Present"].map({'Yes': 1, 'No': 0})
targets = test["Manipulative"].astype(int).tolist()
manipulative_preds = test["Manipulation_Present"].astype(int).tolist()

task1_acc  = accuracy_score(targets, manipulative_preds)
task1_prec = precision_score(targets, manipulative_preds, average="binary", zero_division=0)
task1_rec  = recall_score(targets, manipulative_preds, average="binary", zero_division=0)
task1_f1   = f1_score(targets, manipulative_preds, average="binary", zero_division=0)
task1_cm   = confusion_matrix(targets, manipulative_preds)

print(f"Accuracy:  {task1_acc:.4f}")
print(f"Precision: {task1_prec:.4f}")
print(f"Recall:    {task1_rec:.4f}")
print(f"F1 Score:  {task1_f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"              No      Yes")
print(f"Actual No   {task1_cm[0][0]:4d}    {task1_cm[0][1]:4d}")
print(f"Actual Yes  {task1_cm[1][0]:4d}    {task1_cm[1][1]:4d}")

True label distribution:
 Manipulative
1    95
0    60
Name: count, dtype: int64

Predicted label distribution:
 Manipulation_Present
Yes    87
No     68
Name: count, dtype: int64
Accuracy:  0.8968
Precision: 0.9540
Recall:    0.8737
F1 Score:  0.9121

Confusion Matrix:
                Predicted
              No      Yes
Actual No     56       4
Actual Yes    12      83


### Task 2: Primary Manipulator Identification

Đánh giá khả năng xác định manipulator (Plaintiff/Defendant/None).

In [23]:

print("\n" + "="*50)
print("TASK 2: PRIMARY MANIPULATOR IDENTIFICATION")
print("="*50)

def normalize_manipulator(x):
    if pd.isna(x):
        return "None"
    x = str(x).strip().lower()
    x = x.replace("defendent", "defendant")  # fix typo
    return x.title()  # → "Defendant", "Plaintiff", "None"

test["Primary_Manipulator_norm"]      = test["Primary_Manipulator"].apply(normalize_manipulator)
test["Primary_Manipulator_pred_norm"] = test["Primary Manipulator"].apply(normalize_manipulator)

print("Ground Truth unique (after norm):", sorted(test["Primary_Manipulator_norm"].unique()))
print("Predicted unique  (after norm):",   sorted(test["Primary_Manipulator_pred_norm"].unique()))

encoder = LabelEncoder()
all_classes = sorted(set(
    test["Primary_Manipulator_norm"].tolist() +
    test["Primary_Manipulator_pred_norm"].tolist()
))
encoder.fit(all_classes)

targets_encoded = encoder.transform(test["Primary_Manipulator_norm"])
preds_encoded   = encoder.transform(test["Primary_Manipulator_pred_norm"])

task2_acc  = accuracy_score(targets_encoded, preds_encoded)
task2_prec = precision_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
task2_rec  = recall_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
task2_f1   = f1_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
task2_cm   = confusion_matrix(targets_encoded, preds_encoded,
                               labels=encoder.transform(all_classes))

print(f"\nAccuracy:  {task2_acc:.4f}")
print(f"Precision: {task2_prec:.4f} (weighted)")
print(f"Recall:    {task2_rec:.4f} (weighted)")
print(f"F1 Score:  {task2_f1:.4f} (weighted)")
print(f"\nConfusion Matrix (rows=GT, cols=Pred):")
print(f"Classes: {all_classes}")
print(task2_cm)



TASK 2: PRIMARY MANIPULATOR IDENTIFICATION
Ground Truth unique (after norm): ['Defendant', 'None', 'Plaintiff']
Predicted unique  (after norm): ['Defendant', 'None', 'Plaintiff']

Accuracy:  0.7613
Precision: 0.7708 (weighted)
Recall:    0.7613 (weighted)
F1 Score:  0.7619 (weighted)

Confusion Matrix (rows=GT, cols=Pred):
Classes: ['Defendant', 'None', 'Plaintiff']
[[44  2  6]
 [ 4 56  8]
 [15  2 18]]


### Task 3: Techniques Manipulator Identification

Đánh giá khả năng xác định các kỹ thuật manipulator (11 kỹ thuật).

In [24]:
print("\n" + "="*50)
print("TASK 3: MANIPULATION TECHNIQUES CLASSIFICATION")
print("="*50)

TECHNIQUE_CANON = {
    # typos
    "plaing the victiim":            "playing the victim",
    "minimization.":                 "minimization",
    # variants → chuẩn
    "emotional appealing":           "emotional appeal",
    "emotional framing":             "emotional appeal",
    "guilt":                         "guilt tripping",
    "guilt-tripping":                "guilt tripping",
    "playing the victim role":       "playing the victim",
    "exaggeration":                  "exaggeration",
    "coercion":                      "coercion",
    "character attack":              "character attack",
}

TECHNIQUE_SPLIT = {
    "persuasion emotional appeal":   ["persuasion", "emotional appeal"],
    "deflection minimization":       ["deflection", "minimization"],
    "deflection playing the victim": ["deflection", "playing the victim"],
}

def parse_and_normalize_techniques(x):
    if pd.isna(x) or str(x).strip() in ("None", "nan", ""):
        return ["none"]
    parts = [t.strip().lower() for t in str(x).split(",") if t.strip()]
    result = []
    for p in parts:
        if p in TECHNIQUE_SPLIT:
            result.extend(TECHNIQUE_SPLIT[p])
        else:
            normalized = TECHNIQUE_CANON.get(p, p)
            result.append(normalized)
    return result or ["none"]

test["gt_techniques"]   = test["Manipulation Techniques"].apply(parse_and_normalize_techniques)
test["pred_techniques"] = test["Techniques_Used"].apply(parse_and_normalize_techniques)

unique_gt   = set(t for lst in test["gt_techniques"]   for t in lst)
unique_pred = set(t for lst in test["pred_techniques"] for t in lst)
all_labels  = sorted(unique_gt | unique_pred)

mlb = MultiLabelBinarizer(classes=all_labels)
mlb.fit([all_labels])
y_true = mlb.transform(test["gt_techniques"])
y_pred = mlb.transform(test["pred_techniques"])

task3_acc     = accuracy_score(y_true, y_pred)
task3_prec    = precision_score(y_true, y_pred, average="macro", zero_division=0)
task3_rec     = recall_score(y_true, y_pred, average="macro", zero_division=0)
task3_f1      = f1_score(y_true, y_pred, average="macro", zero_division=0)
task3_jaccard = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

print(f"\nAccuracy:      {task3_acc:.4f} (exact match)")
print(f"Precision:     {task3_prec:.4f} (macro)")
print(f"Recall:        {task3_rec:.4f} (macro)")
print(f"F1 Score:      {task3_f1:.4f} (macro)")
print(f"Jaccard Score: {task3_jaccard:.4f} (samples)")


TASK 3: MANIPULATION TECHNIQUES CLASSIFICATION

Accuracy:      0.3613 (exact match)
Precision:     0.2486 (macro)
Recall:        0.5726 (macro)
F1 Score:      0.3268 (macro)
Jaccard Score: 0.5222 (samples)


### Final Summary

Tổng hợp tất cả metrics từ 3 tasks.

In [25]:
print("\n" + "="*60)
print(" "*15 + "FINAL EVALUATION SUMMARY")
print("="*60)
print(f"\nDataset Size: {len(test)}")
print("\n" + "-"*60)
print(f"{'Task':<45} {'Metric':<12} {'Score':<10}")
print("-"*60)

print(f"{'1. Manipulation Detection (Binary)':<45} {'Accuracy':<12} {task1_acc:.4f}")
print(f"{'':<45} {'Precision':<12} {task1_prec:.4f}")
print(f"{'':<45} {'Recall':<12} {task1_rec:.4f}")
print(f"{'':<45} {'F1':<12} {task1_f1:.4f}")

print(f"{'2. Primary Manipulator (3-class)':<45} {'Accuracy':<12} {task2_acc:.4f}")
print(f"{'':<45} {'Precision':<12} {task2_prec:.4f}")
print(f"{'':<45} {'Recall':<12} {task2_rec:.4f}")
print(f"{'':<45} {'F1 (wtd)':<12} {task2_f1:.4f}")

print(f"{'3. Techniques (Multi-label)':<45} {'Accuracy':<12} {task3_acc:.4f}")
print(f"{'':<45} {'Precision':<12} {task3_prec:.4f}")
print(f"{'':<45} {'Recall':<12} {task3_rec:.4f}")
print(f"{'':<45} {'F1 (macro)':<12} {task3_f1:.4f}")
print(f"{'':<45} {'Jaccard':<12} {task3_jaccard:.4f}")

print("="*60)
print("\nEvaluation completed!")


               FINAL EVALUATION SUMMARY

Dataset Size: 155

------------------------------------------------------------
Task                                          Metric       Score     
------------------------------------------------------------
1. Manipulation Detection (Binary)            Accuracy     0.8968
                                              Precision    0.9540
                                              Recall       0.8737
                                              F1           0.9121
2. Primary Manipulator (3-class)              Accuracy     0.7613
                                              Precision    0.7708
                                              Recall       0.7613
                                              F1 (wtd)     0.7619
3. Techniques (Multi-label)                   Accuracy     0.3613
                                              Precision    0.2486
                                              Recall       0.5726
                     